<a href="https://colab.research.google.com/github/alurusaivahini-dotcom/Trend-pulse/blob/main/task1_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# importing libraries
import requests
import json
import os
import time
from datetime import datetime
# API,URL and HEADER
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"
headers = {"User-Agent": "TrendPulse/1.0"}
# Taking the categories and keywords
CATEGORIES = {
    "technology": [
        "ai", "software", "tech", "code", "computer",
        "data", "cloud", "api", "gpu", "llm"
    ],
    "worldnews": [
        "war", "government", "country", "president",
        "election", "climate", "attack", "global"
    ],
    "sports": [
        "nfl", "nba", "fifa", "sport", "game",
        "team", "player", "league", "championship"
    ],
    "science": [
        "research", "study", "space", "physics",
        "biology", "discovery", "nasa", "genome"
    ],
    "entertainment": [
        "movie", "film", "music", "netflix",
        "game", "book", "show", "award", "streaming"
    ]
}
# Fetch top story Ids
try:
    response = requests.get(TOP_STORIES_URL, headers=headers)
    response.raise_for_status()
    top_story_ids = response.json()[:500]
except Exception as e:
    print("Failed to fetch top stories:", e)
    top_story_ids = []
# collection of stories
all_stories = []
for category, keywords in CATEGORIES.items():
    print(f"\nCollecting stories for category: {category}")
    category_count = 0
    for story_id in top_story_ids:
        if category_count >= 25:
            break
        try:
            story_response = requests.get(
                ITEM_URL.format(story_id),
                headers=headers
            )
            if story_response.status_code != 200:
                print(f"Failed to fetch story {story_id}")
                continue
            story = story_response.json()
            # skip if no title
            if not story or "title" not in story:
                continue
            title_lower = story["title"].lower()
            # check keyword match
            if any(keyword in title_lower for keyword in keywords):
                story_data = {
                    "post_id": story.get("id"),
                    "title": story.get("title"),
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }
                all_stories.append(story_data)
                category_count += 1
        except Exception as e:
            print(f"Error fetching story {story_id}: {e}")
            continue
      # wait after 2 seconds of categories
    time.sleep(2)
    # Save the data in json files
os.makedirs("data", exist_ok=True)
date_str = datetime.now().strftime("%Y%m%d")
file_path = f"data/trends_{date_str}.json"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(all_stories, f, indent=4)
print(f"\nCollected {len(all_stories)} stories. Saved to {file_path}")







Collected 101 stories. Saved to data/trends_20260408.json
